In [1]:
!mkdir -p DATOS
!rm -f ./DATOS/persons.csv
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv" -P ./DATOS
!head ./DATOS/persons.csv

--2026-11-20 02:45:05--  https://github.com/josepgarcia/datos/raw/refs/heads/main/persons.csv
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/persons.csv [following]
--2026-11-20 02:45:06--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/persons.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607 [text/plain]
Saving to: ‘./DATOS/persons.csv’

persons.csv         100%[===================>]     607  --.-KB/s    in 0s      

2026-11-20 02:45:06 (19,3 MB/s) - ‘./DATOS/persons.csv’ saved [607/607]

name,age,gender,salary
John Doe,30,Male,5000

In [2]:
import os, cloudpickle, pyspark.serializers
pyspark.serializers.cloudpickle = cloudpickle
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ejercicio5_CSV").getOrCreate()
sc = spark.sparkContext

# 2. Cargar con sc.textFile() usando ruta absoluta con file://
ruta_csv = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd_raw = sc.textFile(ruta_csv)

# 3. Eliminar el encabezado
encabezado = rdd_raw.first()
rdd_sin_header = rdd_raw.filter(lambda linea: linea != encabezado and linea.strip() != "")

# 4. Extraer solo nombres y edades (suponiendo formato name,age,city)
def extraer_nombre_edad(linea):
    try:
        partes = linea.strip().split(",")
        return (partes[0].strip(), int(partes[1].strip()))
    except (ValueError, IndexError):
        return None

nombres_edades = rdd_sin_header.map(extraer_nombre_edad).filter(lambda x: x is not None)

# 5. Filtrar personas mayores de 30 años
mayores_30 = nombres_edades.filter(lambda x: x[1] > 30)

# 6. Contar y mostrar resultados
total_mayores_30 = mayores_30.count()

print("Encabezado eliminado:", encabezado)
print("Personas mayores de 30:", ", ".join([f"{n} ({e})" for n, e in mayores_30.collect()]))
print(f"Total: {total_mayores_30} personas")

spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/20 02:48:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Encabezado eliminado: name,age,gender,salary
Personas mayores de 30: David Johnson (35), Michael Wilson (40), Sarah Brown (32), James Martinez (38), William Anderson (33), Karen Hernandez (31), Christopher Taylor (37), Matthew Davis (36), Daniel Miller (34), Linda Martin (39)
Total: 10 personas
